# **Epikus Emojik**

<div style="font-size: 14px; color: #6e8192; line-height: 1.5;">
  <div style="display: flex; align-items: center; gap: 5px; margin-bottom: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🎯</span>
    <span>MI Országos Diákolimpia Válogató</span>
  </div>
  <div style="display: flex; align-items: center; gap: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🧠</span>
    <span>Natural Language Processing</span>
  </div>
  <div style="display: flex; align-items: center; gap: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🏆</span>
    <span>100 pont</span>
  </div>
  <div style="display: flex; align-items: center; gap: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🗓️</span>
    <span>2025. Május 24.</span>
  </div>
</div>

**Név:** [ÍRD IDE A NEVED]

**Versenyzői Azonosító:** [ÍRD IDE A VERSENYZŐI AZONOSÍTÓD]

<img src="https://drive.google.com/uc?export=view&id=1vhGZdZ95KsizBHg8C9UifQkOefDkkF8K" alt="petike" style="width:150px;">

**Emoji Emőke** egy önjelölt filmesztéta, aki megszállottan rajong a mozi világáért és az emojikért. Arra tette fel az életét, hogy minden létező filmcímet egyetlen jól megválasztott **emoji-szekvenciával** (4-5 emoji) foglaljon össze.  
😅 „Ha egy film nem írható le három emojival, nem is érdemli meg, hogy megnézzem!”.

Ebben a feladatban egy nyelvi modellt kell tanítanod, amely **angol filmcímek alapján képes emojisorozatokat generálni**. Ehhez egy meglévő adathalmazt, valamint egy előre betanított modellt (EmojiLM) kapsz, amelyet tovább kell finomhangolnod, majd kiértékelned.

### **A feladat lépései**

1. **Adatbetöltés és felfedezés** (10 pont)  

2. **Adatok tisztítása** (5 pont)  

3. **Modell betöltése** (5 pont)  

4. **Adat előfeldolgozás** (20 pont)  

5. **Baseline értékelés, nem finomhangolt modell** (10 pont)  

6. **Modell tanítása** (25 pont)  

7. **A modell kiértékelése és vizualizáció** (3 pont)  

8. **Az eredmények vizualizációja** (7 pont)

9. **Figyelem Mechanizmus Vizualizációja** (15 pont)

A megoldás során az NLP és generatív nyelvi modellezés eszközeit használjátok arra, hogy a címek mögött rejlő tartalmakat emojikkal kifejezzétek. A végső cél, hogy a modell képes legyen **új filmcímekre is frappáns emoji-montázsokat alkotni**, pont úgy, ahogy Emoji Emőke álmodta meg.

## **Hasznos linkek**

- [Hugging Face Transformers - Dokumentáció](https://huggingface.co/docs/transformers/index)  

- [Hugging Face Datasets - Dokumentáció](https://huggingface.co/docs/datasets/index)  

- [T5 modell - alapelvek](https://arxiv.org/abs/1910.10683)  

- [BLEU score - értelmezés és példa](https://machinelearningmastery.com/calculate-bleu-score-for-text-python/)  

- [Python Regex - Emojik szűrése](https://www.regular-expressions.info/unicode.html)


## **Szükséges Importok**

In [ ]:
!pip install --q evaluate nltk
!pip install --q datasets

import torch
import re
import os
import gc
import matplotlib.pyplot as plt
import time
import nltk
from torch.utils.data import DataLoader
from transformers import get_scheduler, DataCollatorForSeq2Seq
from torch.optim import AdamW
from datasets import Dataset
from tqdm import tqdm
from evaluate import load
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')
bleu = load("bleu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.4 MB/s eta 0:00:00


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## **Adathalmaz Letöltése**

A letöltött `.csv` fájl az alábbi két oszlopot tartalmazza:

- **title**: egy angol nyelvű filmcím (pl. `"Finding Nemo"`, `"The Matrix"`)
- **emojis**: az adott filmcímhez tartozó emoji-sorozat, amely a film tartalmát, hangulatát vagy szereplőit próbálja visszaadni (pl. `"🐠🔍🌊"` vagy `"👓💊🤖"`)

Ezt az adatot fogjátok használni a nyelvi modell tanítására: a cél az, hogy új filmcímek esetén a modell megtanuljon hasonlóan frappáns emojikat javasolni.

In [ ]:
!gdown 19d_GJ-DLoTiQegV-oPNIIuDlk_UAcuRt

Downloading...
From: https://drive.google.com/uc?id=19d_GJ-DLoTiQegV-oPNIIuDlk_UAcuRt
To: /content/emoji_dataset.csv
100% 41.2k/41.2k [00:00<00:00, 107MB/s]


### **Adathalmaz Betöltése**

In [ ]:
import pandas as pd

dataset = pd.read_csv("emoji_dataset.csv")

In [ ]:
dataset.head()

,title,emojis
0,The Shawshank Redemption,NaN
1,The Godfather,👨‍👦🕴️🤵🔫🇮🇹
2,The Dark Knight,🦇🤡🃏🚓🚨
3,The Godfather Part II,🧓👶🔫💼🇮🇹
4,12 Angry Men,👨‍⚖️🔍🤔🔒👥


# **1. Feladat (10 pont)**

Válogassunk ki az adathalmazból **10 véletlenszerű példát**, és vizsgáljuk meg, hogyan néznek ki az adatok.

Ezzel a lépéssel bepillantást nyerhetünk abba, hogy milyen jellegű filmcímek és hozzájuk tartozó emojik szerepelnek a tanítóhalmazban. Érdemes megfigyelni, hogy mennyire egyértelmű a kapcsolat a cím és az emoji-sorozat között, például visszautal-e a műfajra, karakterekre vagy eseményekre.

Vizsgáljuk meg, hogy hány különböző filmcím és hány egyedi emoji szerepel az adathalmazban.

- A filmcímek esetében az ismétlődő sorokat kiszűrve számoljuk meg az egyedi címeket.
- Az emojiknál viszont nem az egész emoji-sorozatok számítanak, hanem az összes előforduló egyedi emoji karakter, tehát minden külön emoji egy külön egység.

In [ ]:
# --------- IDE KERÜL A SAJÁT MEGOLDÁSOD --------- (~14-20 sor)

# ----------- ITT A MEGOLDÁSOD VÉGE -------------

# **2. Feladat (5 pont)**

Először tisztítsuk meg az adatainkat:

- Töröljük azokat a sorokat, ahol hiányzik a filmcím vagy az emojik.
- Távolítsuk el az esetleges duplikált sorokat, hogy a modell ne ugyanazt az információt tanulja meg többször.

Ezután válasszunk ki az így megtisztított adathalmazból 60 véletlenszerű példát, amelyekkel a modellt tanítani fogjuk.

In [ ]:
# --------- IDE KERÜL A SAJÁT MEGOLDÁSOD --------- (~3-5 sor)

# ----------- ITT A MEGOLDÁSOD VÉGE -------------

<details>
<summary><strong>💥 Ha a Colab figyelmeztet, hogy megtelt a GPU memória, kattints ide és futtasd ezt a cellát!</strong></summary>

Ez a segédfüggvény felszabadítja a GPU-t, törli a felesleges változókat, üríti a gyorsítótárat, és megpróbálja visszaadni a memóriát a rendszernek.  
❗ **Csak akkor használd, ha a rendszer figyelmeztet vagy hibaüzenet jelenik meg a memóriahasználat miatt!**

```python
def clear_memory():
    if "inputs" in globals():
        del globals()["inputs"]
    if "model" in globals():
        del globals()["model"]
    if "processor" in globals():
        del globals()["processor"]
    if "trainer" in globals():
        del globals()["trainer"]
    if "peft_model" in globals():
        del globals()["peft_model"]
    if "bnb_config" in globals():
        del globals()["bnb_config"]
    time.sleep(2)
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

clear_memory()
```
</details>

# **3. Feladat (5 pont)**

Töltsd be a megadott előre betanított nyelvi modellt a Hugging Face modellközpontból:  
🔗 [`KomeijiForce/t5-base-emojilm`](https://huggingface.co/KomeijiForce/t5-base-emojilm)

Ez egy olyan T5-alapú modell, amelyet kifejezetten emojikkal való szövegfordításra tanítottak. A modell azt várja, hogy a bemenet egy *translate into emojis:* előtaggal kezdődjön, ezt követi a filmcím.

A betöltéshez szükséged lesz a következő komponensekre:
- **Tokenizer**: feldolgozza a bemenetet a modell számára,
- **Generative modell**: maga a betanított T5 modell, amely emojikat állít elő.

In [ ]:
# --------- IDE KERÜL A SAJÁT MEGOLDÁSOD --------- (~4-5 sor)

# ----------- ITT A MEGOLDÁSOD VÉGE -------------

# **4. Feladat (20 pont)**

A modell tanításához az adatokat megfelelő formátumba kell hozni. Ehhez kétféleképpen készítheted elő az adathalmazt:

- Hozz létre egy `Dataset` objektumot (pl. a Hugging Face `datasets` könyvtárral), majd írd meg a saját tokenizáló függvényedet, amely előkészíti a bemeneteket és címkéket.
- VAGY hozz létre egy `DataLoader`-t, amelyhez használhatsz `collate_fn`-t és segédfüggvényeket.

**Fontos technikai szabályok:**
- A bemeneteket egységes hosszúságra kell kiegészíteni, hogy batch-ben lehessen őket feldolgozni.
- A kimeneti címkék esetében a padding tokeneket `-100` értékre kell állítani, hogy a veszteségfüggvény figyelmen kívül hagyja őket.

In [ ]:
# --------- IDE KERÜL A SAJÁT MEGOLDÁSOD --------- (~12-20 sor)

# ----------- ITT A MEGOLDÁSOD VÉGE -------------

# **5. Feladat (10 pont)**

Mielőtt bármit tanítanánk, mérjük fel, hogy **mit tud az alapértelmezett, nem finomhangolt modell**. Ehhez a következőket kell elkészítened:

Írj két segédfüggvényt:
- Az első egy-egy filmcímet ad át a modellnek, és visszaadja az erre generált emojisorozatot.
- A második végigmegy az egész tanítókészleten, a valódi és generált emojisorozatokat összevetve pedig kiszámolja a BLEU pontszámot.

A **BLEU** egy automatikus kiértékelő metrika, amit gyakran használnak nyelvi modellek teljesítményének mérésére, minél magasabb, annál jobb a generált szöveg (jelen esetben emoji-sorozat) egyezése az elvárttal.


In [ ]:
# --------- IDE KERÜL A SAJÁT MEGOLDÁSOD --------- (~18-25 sor)

# ----------- ITT A MEGOLDÁSOD VÉGE -------------

# **6. Feladat (25 pont)**

Ebben a feladatban a cél, hogy saját kóddal vagy a Hugging Face `Trainer` osztály segítségével finomhangold az EmojiLM modellt a kiválasztott mintán.

A tanítás során alkalmaznod kell az ún. **early stopping** technikát, amely megszakítja a tanulást, ha az nem javul tovább.  

Mivel kis adathalmazzal dolgozunk, most nincs validációs készlet, ezért az early stopping-ot a **tanítóhalmaz loss értékére** kell alapozni.

**Két lehetséges megközelítés:**
- Írd meg a tanítókört kézzel (`train loop`), ahol saját magad állítod be:
  - az optimalizálót,
  - a tanulási rátát (learning rate),
  - az ütemezőt (scheduler),
  - a `loss` nyomon követését epochonként.
- Használhatod a Hugging Face `Trainer` osztályát, ha azt megfelelően konfigurálod (pl. `EarlyStoppingCallback`, `TrainingArguments`).

**Loss értékek mentése:**  
A tanulás során minden epoch végén **mentsd el az aktuális train loss értéket**, hogy később vizualizálni lehessen a tanulási görbét.

---

### **Pontozás**

A megszerezhető pontszám a modell tanítási teljesítményétől (loss) függ:

| Elért train loss érték | Megszerezhető pont |
|------------------------|---------------------|
| < 1                    | 25 pont             |
| < 2                    | 20 pont             |
| < 4                    | 15 pont             |
| < 5                    | 10 pont             |

In [ ]:
# --------- IDE KERÜL A SAJÁT MEGOLDÁSOD --------- (~25-35 sor)

# ----------- ITT A MEGOLDÁSOD VÉGE -------------

# **7. Feladat (3 pont)**

Most, hogy betanítottad a modellt, nézzük meg, hogyan teljesít konkrét példákon!

Futtasd le a modellt az adathalmaz első 10 filmcímén, és írd ki:

- az eredeti filmcímet,
- a modell által generált emoji-sorozatot,
- valamint az adott példához tartozó valódi (elvárt) emojikat.

In [ ]:
# --------- IDE KERÜL A SAJÁT MEGOLDÁSOD --------- (~5-6 sor)

# ----------- ITT A MEGOLDÁSOD VÉGE -------------

# **8. Feladat (7 pont)**

Most értékeljük ki, mennyit fejlődött a modell a tanítás során.

A teendőid:

1. Futtasd le a tanított modellt az első 10 filmcímen (ugyanúgy, mint korábban).
   - Hasonlítsd össze a generált emojikat a valódiakkal.

2. Készíts egy ábrát, amely megmutatja a tanítás során rögzített `train loss` értékeket epochonként.

3. Számítsd ki a BLEU pontszámot a tanított modellre is.
   - Hasonlítsd össze az értéket a korábban mért (nem tanított) modell BLEU pontszámával.

In [ ]:
# --------- IDE KERÜL A SAJÁT MEGOLDÁSOD --------- (~8-15 sor)

# ----------- ITT A MEGOLDÁSOD VÉGE -------------

# **9. Feladat (15 pont)**

Miután betanítottad a T5-alapú modellt az adott cím-emojipár adatokon, vizualizáld, hogyan működik a figyelmi mechanizmus (attention). Ehhez válassz ki néhány példát a teszthalmazból, és készíts vizualizációt, amely megmutatja, hogy a generált emojik mely bemeneti címszavakra figyelnek leginkább a dekóder figyelemrétegeiben.

Használj hőtérképes megjelenítést a dekóder-enkóder figyelmi súlyok (cross-attention) ábrázolására.

Követelmények:
- Válassz ki legalább 5 példát a teszthalmazból.
- Használj figyelemsúlyokat (`cross_attentions`) a T5 modellből.
- Készíts legalább 5 hőtérképet (pl. `matplotlib` vagy `seaborn` segítségével).

In [ ]:
# --------- IDE KERÜL A SAJÁT MEGOLDÁSOD --------- (~15-25 sor)

# ----------- ITT A MEGOLDÁSOD VÉGE -------------

---

## 🎉 Gratulálunk!

A feladatsor végére értél - kiváló munkát végeztél!  

---